# ML Assignment 2 — Credit Card Default Classification
**Dataset:** Default of Credit Card Clients (30,000 rows, 23 features, binary target)
**Models:** Logistic Regression, Decision Tree, kNN, Naive Bayes, Random Forest (Ensemble)

**Instructions for use in Google Colab:**
1. Upload `credit_card_default.csv` using the file upload cell below (or mount Google Drive)
2. Run all cells in order
3. Download the `model/` folder and `test_data.csv` at the end — these go into your GitHub repo


## 1. Setup

In [ ]:
!pip install -q scikit-learn pandas numpy joblib


In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.makedirs('model', exist_ok=True)


## 2. Upload dataset
Run this cell and select `credit_card_default.csv` from your computer.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select credit_card_default.csv


## 3. Load and inspect data

In [ ]:
df = pd.read_csv('credit_card_default.csv')

# Drop stray index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

TARGET = 'default_payment_next_month'

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:\n", df[TARGET].value_counts())
df.head()


### Quick EDA — class balance and a couple of feature distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue','indianred'])
axes[0].set_title('Target class balance (0=No Default, 1=Default)')

sns.histplot(df['limit_bal'], bins=40, ax=axes[1], color='steelblue')
axes[1].set_title('Credit Limit Distribution')

sns.histplot(df['age'].dropna(), bins=30, ax=axes[2], color='steelblue')
axes[2].set_title('Age Distribution')

plt.tight_layout()
plt.savefig('model/eda_overview.png', dpi=100)
plt.show()


## 4. Preprocessing
- Impute missing values (median for numeric, most frequent for categorical)
- One-hot encode categorical columns
- Train/test split (80/20, stratified)
- Standard-scale features (used by Logistic Regression and kNN)


In [ ]:
categorical_cols = ['sex', 'education', 'marriage'] + [c for c in df.columns if 'payment_status' in c]
numeric_cols = [c for c in df.columns if c not in categorical_cols + [TARGET]]

df[numeric_cols] = SimpleImputer(strategy='median').fit_transform(df[numeric_cols])
df[categorical_cols] = SimpleImputer(strategy='most_frequent').fit_transform(df[categorical_cols])

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]
feature_names = X.columns.tolist()

print(f"Total features after encoding: {len(feature_names)}")


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train shape:", X_train.shape, " Test shape:", X_test.shape)


## 5. Save test_data.csv (for GitHub repo + Streamlit app upload)
Stratified sample of 1000 rows, kept small for Streamlit free tier.

In [ ]:
test_indices = X_test.index
test_data_raw = df.loc[test_indices].copy()

# Stratified sample of 1000 rows (train_test_split avoids groupby().apply()
# silently dropping the target/group column, which happens in newer pandas)
sample_frac = min(1000 / len(test_data_raw), 1.0)
_, test_data_sample = train_test_split(
    test_data_raw, test_size=sample_frac, random_state=42, stratify=test_data_raw[TARGET]
)
assert TARGET in test_data_sample.columns, "Target column missing from sample!"
test_data_sample.to_csv('test_data.csv', index=False)
print(f"Saved test_data.csv with {len(test_data_sample)} rows")
test_data_sample.head()


## 6. Train all 5 models and compute metrics
Accuracy, AUC, Precision, Recall, F1, MCC — as required by the assignment.


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=8, random_state=42),
    'kNN': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
}

needs_scaling = {'Logistic Regression', 'kNN'}

results = []
confusion_matrices = {}

for name, model in models.items():
    Xtr, Xte = (X_train_scaled, X_test_scaled) if name in needs_scaling else (X_train, X_test)

    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_proba = model.predict_proba(Xte)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)

    results.append({
        'Model': name, 'Accuracy': round(acc,4), 'AUC': round(auc,4),
        'Precision': round(prec,4), 'Recall': round(rec,4),
        'F1': round(f1,4), 'MCC': round(mcc,4)
    })
    confusion_matrices[name] = confusion_matrix(y_test, y_pred).tolist()

    safe_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    with open(f'model/{safe_name}.pkl', 'wb') as f:
        pickle.dump(model, f)

    print(f"{name:30s} Acc={acc:.4f}  AUC={auc:.4f}  F1={f1:.4f}  MCC={mcc:.4f}")

with open('model/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('model/feature_names.json', 'w') as f:
    json.dump(feature_names, f)
with open('model/categorical_cols.json', 'w') as f:
    json.dump(categorical_cols, f)
with open('model/numeric_cols.json', 'w') as f:
    json.dump(numeric_cols, f)


## 7. Comparison table (for your README.md)

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv('model/comparison_table.csv', index=False)
results_df


## 8. Confusion matrices — visual check for each model

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, (name, cm) in zip(axes, confusion_matrices.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('model/confusion_matrices.png', dpi=100)
plt.show()

with open('model/confusion_matrices.json', 'w') as f:
    json.dump(confusion_matrices, f)


## 9. Download everything for your GitHub repo
This zips the `model/` folder (trained models + scaler + metadata) and `test_data.csv` for you to download and add to your repository.


In [ ]:
import shutil
shutil.make_archive('model_and_data', 'zip', '.', 'model')

from google.colab import files
files.download('model_and_data.zip')
files.download('test_data.csv')


## Notes for your README.md

- **Naive Bayes** shows noticeably lower accuracy (~39%) than the other models — this is a genuine, explainable
  result: GaussianNB assumes feature independence, which breaks down here because payment/bill features across
  months are highly correlated. It also shows the highest **recall** — it over-predicts the default class.
- **Random Forest** typically achieves the best AUC, reflecting its strength with non-linear feature interactions.
- **Logistic Regression** and **Decision Tree** perform comparably on accuracy but Logistic Regression edges ahead on MCC.
- Write your own 2-3 sentence observation per model in the README, using the numbers above as evidence —
  this is what your grader is checking for (not just the table).
